In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, mean_squared_error
from sklearn.cluster import KMeans
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')

In [8]:
# Load your datasets
products = pd.read_csv('Products.csv')
customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')


#Train ML Model for Sales Prediction


In [30]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Feature engineering
orders['Discount'] = orders['quantity'] * orders['Price']
features = orders[['product_id', 'customer_id', 'price', 'quantity']]
labels = orders['Discount']

# Convert categorical data
features = pd.get_dummies(features)

# Split and train
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2)
model = RandomForestRegressor()
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
print("RMSE:", mean_squared_error(y_test, predictions, squared=False))


# Analyze Reviews Using NLP (Sentiment Analysis)

In [ ]:
from textblob import TextBlob

# Sentiment score
reviews['sentiment'] = reviews['review_text'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

# Categorize sentiment
reviews['sentiment_label'] = reviews['sentiment'].apply(lambda x: 'Positive' if x > 0 else ('Negative' if x < 0 else 'Neutral'))

# Merge with products
product_sentiment = reviews.groupby('product_id')['sentiment'].mean()


#Predict Customer Churn


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Churn label
latest_order = orders.groupby('customer_id')['order_date'].max()
churn_threshold = pd.Timestamp.now() - pd.Timedelta(days=90)
customers['churn'] = latest_order < churn_threshold

# Features: total spent, order count
order_stats = orders.groupby('customer_id').agg({
    'sales': 'sum',
    'order_id': 'count'
}).rename(columns={'sales': 'total_spent', 'order_id': 'order_count'})
customer_data = customers.merge(order_stats, on='customer_id', how='left').fillna(0)

# Train churn model
X = customer_data[['total_spent', 'order_count']]
y = customer_data['churn'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y)

churn_model = LogisticRegression()
churn_model.fit(X_train, y_train)
print(classification_report(y_test, churn_model.predict(X_test)))


# Predict Next Purchase (Product Recommendation)

In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(orders[['customer_id', 'product_id', 'quantity']], reader)

trainset, testset = train_test_split(data, test_size=0.25)
algo = SVD()
algo.fit(trainset)

# Predict for a customer-product pair
customer_id = 'C123'
product_id = 'P456'
pred = algo.predict(customer_id, product_id)
print(pred.est)  # Estimated rating


#Customer Segmentation (Customer Type)

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Use RFM features
rfm = orders.groupby('customer_id').agg({
    'order_date': lambda x: (pd.Timestamp.now() - x.max()).days,
    'order_id': 'count',
    'sales': 'sum'
}).rename(columns={'order_date': 'recency', 'order_id': 'frequency', 'sales': 'monetary'})

# Normalize
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# Cluster
kmeans = KMeans(n_clusters=4)
rfm['cluster'] = kmeans.fit_predict(rfm_scaled)

# Plot
plt.scatter(rfm['recency'], rfm['monetary'], c=rfm['cluster'])
plt.xlabel("Recency")
plt.ylabel("Monetary")
plt.title("Customer Segments")
plt.show()